#  Data Collection & Understanding


Data Sources
Traffic accident databases (police reports and News articles)

Target Variable:
Severity (e.g., Critical, Fatal, Serious, Minor)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [26]:
# Load data
df = pd.read_csv('data/dataset.csv')

# Initial Exploration Process

In [27]:
print(df.head())

   person_id  article_id   age  gender severity is_driver vehicle_type  \
0          0        4208  78.0    male     none      True          car   
1          1        4093  54.0    male  serious      True   motorcycle   
2          2        4110  54.0    male  serious      True   motorcycle   
3          3        4066  17.0  female  serious     False          NaN   
4          4        4066  52.0    male     none      True          car   

  person_state  accident_id            date_time  ... windspeed_10m  \
0          NaN          0.0  2024-12-04 17:00:00  ...           9.8   
1          NaN          1.0  2024-12-11 17:00:00  ...           6.5   
2          NaN          2.0  2024-12-11 13:00:00  ...           5.0   
3          NaN          3.0  2024-12-13 17:30:00  ...          24.0   
4          NaN          3.0  2024-12-13 17:30:00  ...          24.0   

  cloud_cover  weather_code  is_weekday  is_public_holiday  time_of_day  \
0       100.0           3.0        True              

In [28]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   person_id          682 non-null    int64  
 1   article_id         682 non-null    int64  
 2   age                618 non-null    float64
 3   gender             581 non-null    object 
 4   severity           661 non-null    object 
 5   is_driver          678 non-null    object 
 6   vehicle_type       568 non-null    object 
 7   person_state       17 non-null     object 
 8   accident_id        682 non-null    float64
 9   date_time          682 non-null    object 
 10  accident_type      682 non-null    object 
 11  location           643 non-null    object 
 12  temperature_2m     682 non-null    float64
 13  precipitation      682 non-null    float64
 14  windspeed_10m      682 non-null    float64
 15  cloud_cover        682 non-null    float64
 16  weather_code       682 non

In [29]:
print(df.describe())

        person_id     article_id         age  accident_id  temperature_2m  \
count  682.000000     682.000000  618.000000   682.000000      682.000000   
mean   371.187683  317307.048387   43.571197   180.806452       21.182551   
std    212.911169  230471.115197   19.330335   103.351108        5.433756   
min      0.000000     287.000000    2.000000     0.000000        9.800000   
25%    184.250000    3700.250000   28.000000    87.000000       15.900000   
50%    376.500000  491011.000000   40.000000   184.000000       21.800000   
75%    556.750000  496803.000000   58.000000   269.000000       25.500000   
max    728.000000  496883.000000   87.000000   354.000000       33.700000   

       precipitation  windspeed_10m  cloud_cover  weather_code       month  \
count     682.000000     682.000000   682.000000     682.00000  682.000000   
mean        0.034751      14.738416    30.065982       5.88563    6.563050   
std         0.223923       8.872948    38.797036      15.37677    3.2663

In [30]:
print(df['severity'].value_counts())

severity
none        280
serious     254
fatal        82
minor        44
critical      1
Name: count, dtype: int64


# Data Processing

In [31]:
# Check missing values
print(df.isnull().sum())

person_id              0
article_id             0
age                   64
gender               101
severity              21
is_driver              4
vehicle_type         114
person_state         665
accident_id            0
date_time              0
accident_type          0
location              39
temperature_2m         0
precipitation          0
windspeed_10m          0
cloud_cover            0
weather_code           0
is_weekday             0
is_public_holiday      0
time_of_day           80
month                  0
hour                  80
day                    0
year                   0
dtype: int64


Removing unnecessary columns.

In [32]:
df.drop('person_id', inplace=True, axis=1)
df.drop('article_id', inplace=True, axis=1)
df.drop('accident_id', inplace=True, axis=1)
df.drop('date_time', inplace=True, axis=1)

From the result above for the target variable "Severity", the data is highly imbalanced.

In [48]:
# Impute missing values
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')
df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

In [49]:
print(df.isnull().sum())

age                 0
temperature_2m      0
precipitation       0
windspeed_10m       0
cloud_cover         0
                   ..
location_Xewkija    0
location_Zabbar     0
location_Zebbug     0
location_Zejtun     0
location_Zurrieq    0
Length: 113, dtype: int64


Handling Categorical Variables:

In [57]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: Index(['age', 'temperature_2m', 'precipitation', 'windspeed_10m',
       'cloud_cover', 'weather_code', 'month', 'hour', 'day', 'year',
       'severity_encoded'],
      dtype='object')
Categorical columns: Index(['time_of_day'], dtype='object')


Removing unnecessary columns.

In [58]:
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

In [59]:
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [61]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Assume your dataframe is df

# 1. Ordinal Encoding (Label Encoding with order)

severity_order = ['none', 'fatal','serious', 'critical', 'minor']

severity_map = {k: v for v, k in enumerate(severity_order)}

df['severity'] = df['severity'].map(severity_map)

# Drop original ordinal columns
df.drop(['severity'], axis=1, inplace=True)

# 2. Nominal Encoding (One-Hot Encoding)

nominal_cols = [
    'gender', 'is_driver', 'vehicle_type',
    'person_state', 'accident_type', 'location'
]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print(df.head())

KeyError: 'severity'

In [53]:
feature_names = X.columns.tolist()

Scaling Features(Very Important for SVM)

In [55]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

NameError: name 'y_encoded' is not defined

In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


NameError: name 'X_train' is not defined